# 01 Project Overview

            This notebook is the orientation layer for the Ibadan Urban Heat Risk Intelligence System.
            It checks the project configuration, explains the pipeline order, and confirms which datasets
            already exist before you run heavier geospatial processing.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "notebook_helpers.py").exists():
    sys.path.insert(0, str(cwd))
elif (cwd / "notebooks" / "notebook_helpers.py").exists():
    sys.path.insert(0, str(cwd / "notebooks"))
else:
    raise FileNotFoundError("Could not find notebook_helpers.py. Run this notebook from the project root or notebooks folder.")

from notebook_helpers import (
    PROJECT_ROOT,
    add_project_root_to_path,
    command_string,
    find_files,
    load_yaml_config,
    notebook_metadata,
    path_status,
    plot_raster,
    print_path_status,
    project_path,
    raster_info,
    raster_stats,
    read_vector,
    run_command,
)

add_project_root_to_path()
RUN_COMMANDS = False  # Change to True only when you want notebook cells to execute CLI scripts.
YEAR = 2023
notebook_metadata("Project Overview", YEAR)

## Step 1 - Load the project configuration

The main project settings live in `config/project_config.yml`.

In [ ]:
config = load_yaml_config("config/project_config.yml")
            config

## Step 2 - Review the target LGAs

These are the core and peri-urban LGAs used to define Ibadan Metropolis.

In [ ]:
import pandas as pd

            target_lgas = config["study_area"]["target_lgas"]
            pd.DataFrame({"target_lga": target_lgas})

## Step 3 - Check important project paths

This tells you what has already been generated and what still needs to be prepared.

In [ ]:
import pandas as pd

            important_paths = {
                "raw Nigeria LGA boundary": "data/raw/boundary/nigeria_lgas.shp",
                "prepared Ibadan LGAs": "data/processed/uhi/ibadan_lgas.gpkg",
                "prepared metro boundary": "data/processed/uhi/ibadan_metropolitan_boundary.gpkg",
                "LST raster": f"data/processed/lst/lst_ibadan_{YEAR}_celsius.tif",
                "NDVI raster": f"data/processed/indices/ndvi_{YEAR}.tif",
                "UHI raster": f"data/processed/uhi/uhi_intensity_{YEAR}.tif",
                "LISA clusters": f"data/processed/esda/lisa_clusters_{YEAR}.gpkg",
                "GWR results": f"data/processed/gwr/gwr_results_{YEAR}.gpkg",
                "HVI raster": f"data/processed/vulnerability/heat_vulnerability_index_{YEAR}.tif",
            }
            pd.DataFrame(path_status(important_paths))

## Step 4 - Run or preview the dependency check

The notebook defaults to dry-run mode. Set `RUN_COMMANDS = True` in the setup cell to execute.

In [ ]:
run_command(["python", "scripts/00_check_dependencies.py"], dry_run=not RUN_COMMANDS)

## Step 5 - Pipeline command map

Use this table as a reproducible execution guide.

In [ ]:
commands = [
                ("Dependency check", "python scripts/00_check_dependencies.py"),
                ("Prepare study area", "python scripts/01_prepare_study_area.py --boundary data/raw/boundary/nigeria_lgas.shp"),
                ("Alternative prebuilt boundary import", "python scripts/01b_import_prebuilt_boundary.py --gpkg data/raw/boundary/ibadan_lgas.gpkg"),
                ("GEE Landsat export", "python scripts/02a_gee_export_landsat.py --years 2015 2020 2023"),
                ("Validate Landsat inputs", f"python scripts/02_download_or_prepare_satellite_data.py --year {YEAR}"),
                ("Compute LST", f"python scripts/03_compute_lst.py --year {YEAR}"),
                ("Compute indices", f"python scripts/04_compute_urban_indices.py --year {YEAR}"),
                ("Compute UHI", f"python scripts/05_compute_uhi_intensity.py --year {YEAR} --reference auto"),
                ("Run ESDA", f"python scripts/06_run_esda.py --target lst --year {YEAR}"),
                ("Run GWR", f"python scripts/07_run_gwr.py --year {YEAR} --dependent lst --predictors ndvi ndbi population_density built_up_density"),
                ("Build HVI", f"python scripts/08_build_vulnerability_index.py --weights config/ahp_weights.yml --year {YEAR}"),
                ("Generate outputs", f"python scripts/09_generate_outputs.py --year {YEAR}"),
                ("Temporal comparison", "python scripts/10_temporal_comparison.py"),
            ]
            pd.DataFrame(commands, columns=["stage", "command"])

## Step 6 - Inspect prepared boundary when available

In [ ]:
boundary_path = project_path("data/processed/uhi/ibadan_lgas.gpkg")
            if boundary_path.exists():
                lgas = read_vector(boundary_path)
                display(lgas.head())
                print(lgas.crs)
                ax = lgas.plot(figsize=(8, 8), edgecolor="black", alpha=0.6)
                ax.set_title("Prepared Ibadan LGAs")
                ax.set_axis_off()
            else:
                print("Prepared boundary is not available yet. Run the study area preparation script first.")